# ACS Yearly Socioeconomic Indicators (2010–2024)

**Purpose:** This notebook fetches yearly American Community Survey (ACS) profile data for New York City counties (boroughs), computes derived indicators (population density, poverty, median household income, Medicaid enrollment share, etc.), and saves a compiled CSV ready for analysis.

**Data sources:**
- U.S. Census Bureau ACS 1-year profile API (2010–2024, excluding 2020 where noted).

**Primary outputs:**
- `data_csv/ACS_socioeconomic_indicators_2010-2024_Data.csv` — consolidated borough-year table used in downstream analyses.

**Dependencies:**
- Python 3.8+
- `requests`, `pandas`

**How to run:**
1. Activate the project virtual environment (if present):
```bash
source .venv/bin/activate
```
2. Run this notebook in Jupyter or execute the code cell(s) to fetch and save the CSV. The notebook uses a Census API key — keep it private and do not commit it to public repositories.

**Citation (for academic use):**
Please cite the U.S. Census Bureau ACS as the original data source. Example BibTeX entry:
```bibtex
@misc{uscensus_acs,
  title = {American Community Survey (ACS) 1-Year Estimates},
  author = {{U.S. Census Bureau}},
  year  = {2010--2024},
  note  = {Accessed via the ACS Profile API},
  url   = {https://www.census.gov/programs-surveys/acs}
}
```

**Notes & reproducibility:**
- The notebook intentionally concatenates borough-level county codes and fetches per-year profiles to create a harmonized timeseries. Some ACS variables changed names across years; the code includes small mappings to handle year-aware variable selections.
- Before sharing the repository, remove or rotate any embedded API keys. Consider adding a `.env` or configuration file and document how to supply the key via an environment variable.

**Author / Maintainer:** Mayank Anand — mayank.anand3007@gmail.com

In [1]:
import requests
import pandas as pd

API_KEY = "f6d11e4ea18e6caeef9cf07a4854e6f657c63e35"
STATE = "36"  # New York

BOROUGHS = {
    "061": "Manhattan",
    "047": "Brooklyn",
    "081": "Queens",
    "005": "Bronx",
    "085": "Staten Island"
}

# Land area in square miles for population density (from Census 2020)
BOROUGH_AREA_SQMI = {
    "061": 22.83,    # Manhattan
    "047": 70.82,    # Brooklyn
    "081": 108.53,   # Queens
    "005": 42.10,    # Bronx
    "085": 58.37     # Staten Island
}
# -------------------------------------------------------
# Year-aware variable definitions
# -------------------------------------------------------
def get_variable_map(year):
    """Returns the correct DP05 variable codes for a given year."""
    
    if year <= 2016:
        return {
            "DP03": "DP03_0062E,DP03_0119PE,DP03_0099PE,DP03_0068PE",
            "DP04": "DP04_0078PE",
            "DP05": "DP05_0001E,DP05_0019PE,DP05_0033PE,DP05_0066PE"
        }
    else:  # 2017 onwards
        return {
            "DP03": "DP03_0062E,DP03_0119PE,DP03_0099PE,DP03_0068PE",
            "DP04": "DP04_0078PE",
            "DP05": "DP05_0001E,DP05_0019PE,DP05_0038PE,DP05_0071PE"
        }

def fetch_profile(year, table_vars, label):
    county_list = ",".join(BOROUGHS.keys())
    url = (
        f"https://api.census.gov/data/{year}/acs/acs1/profile"
        f"?get=NAME,{table_vars}"
        f"&for=county:{county_list}"
        f"&in=state:{STATE}"
        f"&key={API_KEY}"
    )
    r = requests.get(url)
    if r.status_code != 200:
        print(f"  ⚠️  {year} {label} failed: {r.status_code}")
        return None
    data = r.json()
    return pd.DataFrame(data[1:], columns=data[0])


all_years = []

for year in range(2010, 2026):
    if year == 2020:
        continue

    print(f"Fetching {year}...")
    var_map = get_variable_map(year)

    df03 = fetch_profile(year, var_map["DP03"], "DP03")
    df04 = fetch_profile(year, var_map["DP04"], "DP04")
    df05 = fetch_profile(year, var_map["DP05"], "DP05")

    if any(df is None for df in [df03, df04, df05]):
        print(f"  Skipping {year}.")
        continue

    # Merge on county
    df = df03.merge(df04[["county", "DP04_0078PE"]], on="county")
    df = df.merge(df05[["county", "DP05_0001E", "DP05_0019PE"]], on="county", how="left")

    df["year"]    = year
    df["borough"] = df["county"].map(BOROUGHS)

    # Convert to numeric (be robust when some DP05 columns are missing for some years)
    num_cols = [
        "DP03_0062E", "DP03_0119PE", "DP03_0099PE",
        "DP03_0068PE", "DP04_0078PE", "DP05_0001E", 
        "DP05_0019PE" 
    ]

    for col in num_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

    # Derived variables
    df["land_area_sqmi"]        = df["county"].map(BOROUGH_AREA_SQMI)
    df["pop_density_per_sqmi"]  = df["DP05_0001E"] / df["land_area_sqmi"]

    all_years.append(df)
    print(f"  ✅ {year} done")

final = pd.concat(all_years, ignore_index=True)

rename_map = {
    "DP03_0062E":  "median_hh_income",
    "DP03_0119PE": "pct_below_poverty",
    "DP03_0099PE": "pct_uninsured",
    "DP03_0068PE": "pct_medicaid",   # NEW
    "DP04_0078PE": "pct_overcrowded_housing",
    "DP05_0001E":  "total_population",
    "DP05_0019PE": "pct_children_under18",
}

rename_map = {k: v for k, v in rename_map.items() if k in final.columns}
final.rename(columns=rename_map, inplace=True)

cols = [
    "year", "borough", "county",
    "median_hh_income", "pct_below_poverty",
    "pct_children_under18", "pct_uninsured",
    "pct_medicaid", "total_population", 
    "pop_density_per_sqmi", "pct_overcrowded_housing"
]
final = final[[c for c in cols if c in final.columns]]

final.to_csv("data_csv/ACS_socioeconomic_indicators_2010-2024_Data.csv", index=False)
print("\n✅ Saved! Preview:")
print(final[["year","borough"]].head(15))

Fetching 2010...
  ✅ 2010 done
Fetching 2011...
  ✅ 2011 done
Fetching 2012...
  ✅ 2012 done
Fetching 2013...
  ✅ 2013 done
Fetching 2014...
  ✅ 2014 done
Fetching 2015...
  ✅ 2015 done
Fetching 2016...
  ✅ 2016 done
Fetching 2017...
  ✅ 2017 done
Fetching 2018...
  ✅ 2018 done
Fetching 2019...
  ✅ 2019 done
Fetching 2021...
  ✅ 2021 done
Fetching 2022...
  ✅ 2022 done
Fetching 2023...
  ✅ 2023 done
Fetching 2024...
  ✅ 2024 done
Fetching 2025...
  ⚠️  2025 DP03 failed: 404
  ⚠️  2025 DP04 failed: 404
  ⚠️  2025 DP05 failed: 404
  Skipping 2025.

✅ Saved! Preview:
    year        borough
0   2010          Bronx
1   2010       Brooklyn
2   2010      Manhattan
3   2010         Queens
4   2010  Staten Island
5   2011          Bronx
6   2011       Brooklyn
7   2011      Manhattan
8   2011         Queens
9   2011  Staten Island
10  2012          Bronx
11  2012       Brooklyn
12  2012  Staten Island
13  2012         Queens
14  2012      Manhattan
